Show and tell のデモ

In [ ]:
"""
image captioning
「SHOW AND TELL」 (LSTM) MS_COCO 2014
"""
import os
import sys
import glob
from typing import (
    Callable,
    Sequence,
    Tuple,
    Union,
    List,
    Dict,
    Optional,
)
import json
import math
import shutil
from pathlib import Path
import random
from collections import (
    deque, Counter
)
import pickle
import datetime
from pprint import pprint

import numpy as np
from scipy.optimize import linear_sum_assignment # ハンガリアンアルゴリズム
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torch.nn.utils import clip_grad_norm
from tqdm import tqdm

# torch
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torch import optim
from torch.nn.utils.rnn import pack_padded_sequence

# torchvision
import torchvision
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from torchvision.datasets import CocoCaptions
from torchvision.utils import (
    draw_bounding_boxes,
    draw_keypoints,
    draw_segmentation_masks,
)
from torchvision.ops import (
    sigmoid_focal_loss,
    batched_nms,
)
from torchvision.ops.misc import FrozenBatchNorm2d
from torchvision import models

# MS_COCO
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO



In [ ]:
from arch import (
    CNNEncoder,
    LSTMDecoder,    
)

In [ ]:
class ConfigDemo:
    def __init__(self):
        self.dim_embedding = 300
        self.dim_hidden = 128
        self.num_layers = 2

        self.img_directory = "F:\\MS_COCO\\2014\\tasks\\image_captioning"
        self.id_to_word_file = os.path.join(os.getcwd(), "coco2014_val_id_to_word.pkl")
        self.save_directory = os.path.join(os.getcwd(), "model")

        self.device = 'cuda'

In [ ]:
def demo():
    config = ConfigDemo()

    # 単語ID → 単語
    with open(config.id_to_word_file, 'rb') as f:
        id_to_word = pickle.load(f)

    vocab_size = len(id_to_word)

    transforms = T.Compose([
        T.Resize((224,224)),
        T.ToTensor(),
        # ImageNet標準化
        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])

    # Encoder
    encoder = CNNEncoder(config.dim_embedding)
    encoder.to(config.device)
    encoder.eval()

    # Decoder
    decoder = LSTMDecoder(config.dim_embedding,
                          config.dim_hidden,
                          vocab_size,
                          config.num_layers)
    decoder.to(config.device)
    decoder.eval()

    # Loading best parameters
    encoder.load_state_dict(f"{config.save_direcotry}/show_and_tell_encoder_best.pth")
    decoder.load_state_dict(f"{config.save_directory}/show_and_tell_decoder_best.pth")

    # Do captionning the images in target directory
    for img_file in sorted(glob.glob(os.path.join(config.img_directory, '*.jpg'))):
        # 画像読み込み
        img = Image.open(img_file)
        img = transforms(img)
        img = img.unsqueeze(dim=0)
        img = img.to(config.device)

        # Prediction
        feature = encoder(img)
        sampled_ids = decoder.sample(feature)

        # Show sample image
        img_plt = Image.open(img_file)
        img_plt = img_plt.resize([224,224], Image.LANCZOS)
        plt.imshow(img_plt)
        plt.axis('off')
        plt.show()
        print(f"入力画像: {os.path.basename(img_file)}")

        # Captioning
        sampled_caption = []
        for word_id in sampled_ids:
            word = id_to_word[word_id]
            sampled_caption.append(word)
            if word == '<end>':
                break

        sentence = ' '.join(sampled_caption)
        print(f"出力キャプション: {sentence}")

        # Writing predicted sentence
        gen_sentence_out = img_file[:-4] + '_show_and_tell.txt'
        with open(gen_sentence_out, 'w') as f:
            print(sentence, file=f)


In [ ]:
demo()